In [8]:
class StaticChans:
    def __init__(self, full_path, laser_power, PMT, filt, master_idx):
        self.full_path = full_path
        self.laser_power = laser_power
        self.PMT = PMT
        self.filt = filt
        self.master_idx = master_idx
        self.img_shape = io.imread(self.full_path[0]).shape
        self.is_zstack = len(self.img_shape)>2
        self.reg_imgs = np.nan
        self.reg_qual = np.nan
        self.ext_reg_imgs = np.nan
        self.ext_reg_qual = np.nan
        self.status = 0
        self.check_sizes()
    
    def check_sizes():
        if len({len(self.full_path), 
                len(self.laser_power), 
                len(self.PMT), 
                len(self.filt)}) != 1:
            print("Warning: mismatched inputs.")
            self.status = 1
            return
        
        for path in self.full_path[1:]:
            if io.imread(path).shape != self.img_shape:
                print("Warning: not all images have the same shape")
                self.status = 1
    
    def register_internal(self):
        if self.is_zstack:
            self._register_internal_zstack()
        else:
            self._register_internal_2D()

    def register_external(self, ref_img):
        if np.isnan(self.reg_imgs):
            print("Error: channels haven't been internally registered yet.")
            self.status = 1
        elif self.is_zstack:
            self._register_external_zstack(ref_img)
        else:
            self._register_external_2D(ref_img)
    
    def _register_internal_zstack(self):
        reg_imgs = []
        num_planes = self.img_shape[0]
        master_stack = io.imread(self.full_path[self.master_idx])
        reg_qual_all = np.full((len(self.full_path), num_planes, num_planes)) # chan idx, chan plane, master plane
        for n in range(num_planes):
            temp = []
            master_img = master_stack[n, :, :]
            for i in range(len(self.full_path)):
                stack = io.imread(self.full_path[i])
                for n2 in range(num_planes):
                    reg_img, _ = img_shift(master_img, stack[n2, :, :])
                    reg_qual_all[i, n2, n] = calc_reg_qual(master_img, reg_img)
                    
                best_qual_img = stack[np.argmax(reg_qual_all[i, :, n].flatten()), :, :]
                reg_img, _ = img_shift(master_img, best_qual_img)
                temp.append(reg_img.flatten())

            temp = np.column_stack(temp) # in final format nPixels x nChannels
            reg_imgs.append(temp) # reg_imgs will eventually have size (nPlanes,)
            
        self.reg_imgs = reg_imgs
        self.reg_qual = reg_qual_all
        
    def _register_internal_2D(self):
        reg_imgs = []
        reg_qual = []
        master_img = io.imread(self.full_path[self.master_idx])
        for path in self.full_path:
            img = io.imread(path)
            reg_img, _ = img_shift(master_img, img)
            reg_imgs.append(reg_img)
            reg_qual.append(calc_reg_qual(master_img, reg_img))

        reg_imgs = np.column_stack(reg_imgs) # in final format nPixels x nChannels
        self.reg_imgs = reg_imgs
        self.reg_qual = np.array(reg_qual)

    def _register_external_zstack(self, ref_img):
        num_planes = self.img_shape[0]
        ext_reg_imgs = [] # same format as reg_imgs
        ext_reg_qual = np.full((nPlanes,), np.nan)
        for n in range(num_planes):
            temp = []
            master_img = (self.reg_imgs[n][:, self.master_idx]).reshape(self.img_shape[1:])
            shifted_master_img, shift_estimate = img_shift(ref_img, master_img)
            ext_reg_qual[n] = calc_reg_qual(ref_img, shifted_master_img)
            for i in range(self.reg_imgs[n].shape[1]):
                img = (self.reg_imgs[n][:, i]).reshape(self.img_shape[1:])
                shifted_img = shift(mov_img, shift=shift_estimate, mode='nearest')
                temp.append(shifted_img.flatten())
            temp = np.column_stack(temp)
            ext_reg_imgs.append(temp)
                
        self.ext_reg_imgs = ext_reg_imgs
        self.ext_reg_qual = ext_reg_qual
    
    def _register_external_2D(self, ref_img):
        ext_reg_imgs = [] # same format as reg_imgs

        master_img = (self.reg_imgs[:, self.master_idx]).reshape(self.img_shape)
        shifted_master_img, shift_estimate = img_shift(ref_img, master_img)
        ext_reg_qual = calc_reg_qual(ref_img, shifted_master_img)
        for i in range(self.reg_imgs.shape[1]): # iterate over all channels
            img = (self.reg_imgs[:, i]).reshape(self.img_shape)
            shifted_img = shift(mov_img, shift=shift_estimate, mode='nearest')
            ext_reg_imgs.append(shifted_img.flatten())
            
        ext_reg_imgs = np.column_stack(ext_reg_imgs) # nPixels x nChannels
       
        self.ext_reg_imgs = ext_reg_imgs
        self.ext_reg_qual = ext_reg_qual
        
    @staticmethod
    def img_shift(ref_img, mov_img):
        shift_estimate, _, _ = phase_cross_correlation(ref_image, mov_img, upsample_factor=10)
        return shift(mov_img, shift=shift_estimate, mode='nearest'), shift_estimate

    @staticmethod
    def calc_reg_qual(ref_img, reg_img):
        return np.corrcoef(reg_img.flatten(), ref_img.flatten())[0, 1]

    # need an option to say whether classify will run on ext_reg_imgs or reg_imgs (maybe I should just have reg_imgs and override?)
    def classify(self, GT_vecs, settings):
        # GT_vecs is nChannels x nFluorophores
        # settings is a dict with script settings, must contain at least "cellpose_diameter", "nLayers_erode", "ROI_frac"

    def run_cellpose(self, settings):
        model = models.Cellpose(model_type='cyto', gpu=True)  # Set gpu=False if no GPU
        
        # Run segmentation; diameter set to None for automatic
        master_image = np.reshape(multi_dict["all reg"][best_plane][:, ref_stack_idx], multi_dict["original shape"])
        res = list(model.eval(master_image, diameter=cellpose_diameter, channels=[0,0])) # change diameter back to None!
    
        # erode ROIs 
        res[0] = erode_rois(labels=res[0], nLayers=nLayers_erode)
        